Gaussian Mixture Models (GMM)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
from pathlib import Path

plt.style.use("seaborn-v0_8")

PATH_IN  = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/processed/fotos_canarias.csv"
OUT_BASE = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_gmm/"
PUNTOS_CSV = OUT_BASE + "dataset_flickr_gmm_puntos.csv"
COMP_CSV   = OUT_BASE + "dataset_flickr_gmm_comportamiento.csv"

import os
os.makedirs(OUT_BASE, exist_ok=True)



In [2]:
df = pd.read_csv(PATH_IN)

df = df.dropna(subset=["latitude", "longitude"]).copy()
df["datetaken"] = pd.to_datetime(df["datetaken"], errors="coerce")
df = df[df["accuracy"].fillna(0) >= 15].copy()
df = df.drop_duplicates(subset=["owner", "latitude", "longitude", "datetaken"])

In [3]:
centroides = (
    df.groupby("isla")[["latitude", "longitude"]]
      .median()
      .to_dict("index")
)

def asignar_isla_por_centroides(lat, lon, centroides):
    best, dmin = None, 1e9
    for isla, coords in centroides.items():
        d = (lat - coords["latitude"])**2 + (lon - coords["longitude"])**2
        if d < dmin:
            best, dmin = isla, d
    return best

if "isla_fix" not in df.columns:
    df["isla_fix"] = df.apply(
        lambda r: r["isla"] if pd.notna(r["isla"])
                  else asignar_isla_por_centroides(r.latitude, r.longitude, centroides),
        axis=1
    )

In [4]:
R_EARTH_M = 6371000.0

def r90_metros(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(np.clip(
        np.sin(lat0)*np.sin(pts[:,0]) +
        np.cos(lat0)*np.cos(pts[:,0])*np.cos(pts[:,1]-lon0),
        -1.0, 1.0
    ))
    return np.percentile(d * R_EARTH_M, 90)



In [5]:
def meters_per_degree(lat_rad):
    mlat = 111132.954 - 559.822*np.cos(2*lat_rad) + 1.175*np.cos(4*lat_rad)
    mlon = (111412.84*np.cos(lat_rad)
            - 93.5*np.cos(3*lat_rad)
            + 0.118*np.cos(5*lat_rad))
    return mlat, mlon

def island_projection(df_isla):
    lat_ref = np.deg2rad(df_isla["latitude"].median())
    lon_ref = np.deg2rad(df_isla["longitude"].median())
    mlat, mlon = meters_per_degree(lat_ref)

    lat = np.deg2rad(df_isla["latitude"].values)
    lon = np.deg2rad(df_isla["longitude"].values)
    x = (lon - lon_ref) * mlon * (180/np.pi)
    y = (lat - lat_ref) * mlat * (180/np.pi)
    return np.column_stack([x, y])



In [6]:
def best_gmm_for_island(XY, island_name, max_k=12):
    bics = []
    models = []

    Ks = list(range(1, max_k+1))
    for k in Ks:
        gm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            max_iter=500,
            random_state=42
        )
        gm.fit(XY)
        bics.append(gm.bic(XY))
        models.append(gm)

    k_opt = Ks[int(np.argmin(bics))]
    model_opt = models[int(np.argmin(bics))]
    labels = model_opt.predict(XY)

    return model_opt, k_opt, labels

In [7]:
df["cluster_gmm"] = -1

best_by_isla = {}

for isla in sorted(df["isla_fix"].dropna().unique()):
    sub = df[df["isla_fix"] == isla]

    XY = island_projection(sub)

    model, k_opt, labels = best_gmm_for_island(XY, isla, max_k=12)

    df.loc[sub.index, "cluster_gmm"] = labels
    best_by_isla[isla] = {
        "k_opt": int(k_opt),
        "covariance_type": model.covariance_type
    }



In [8]:
MIN_CLUSTER_SIZE_FINAL = 10
sizes = df.groupby("cluster_gmm").size()
small = sizes[sizes < MIN_CLUSTER_SIZE_FINAL].index
df.loc[df["cluster_gmm"].isin(small), "cluster_gmm"] = -1



In [9]:
df_export = df.copy()
df_export.to_csv(PUNTOS_CSV, index=False, encoding="utf-8")

if "Tipo_behavior" in df_export.columns:
    comp = (
        df_export[df_export["cluster_gmm"] != -1]
            .groupby(["isla_fix", "cluster_gmm"])["Tipo_behavior"]
            .value_counts(normalize=True)
            .unstack()
            .fillna(0)
    )
    comp.to_csv(COMP_CSV, encoding="utf-8")

print("✓ Exportados puntos y comportamiento en:")
print("-", PUNTOS_CSV)
print("-", COMP_CSV if "Tipo_behavior" in df_export.columns else "(sin comportamiento)")



✓ Exportados puntos y comportamiento en:
- /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_gmm/dataset_flickr_gmm_puntos.csv
- /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_gmm/dataset_flickr_gmm_comportamiento.csv


In [10]:
import pandas as pd
import geopandas as gpd
import numpy as np

from shapely.ops import unary_union
from shapely.geometry import LineString
from pathlib import Path


PUNTOS_CSV = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_gmm/dataset_flickr_gmm_puntos.csv"
OUT_GPKG   = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/gmm/gmm_canarias.gpkg"

CRS_WGS84 = "EPSG:4326"
CRS_UTM28 = "EPSG:32628"


df = pd.read_csv(PUNTOS_CSV)

if "latitude" not in df or "longitude" not in df:
    raise ValueError("El CSV no contiene latitude/longitude.")


def nivel_porcentaje(p):
    if p < 0.25:
        return 1
    elif p < 0.50:
        return 2
    elif p < 0.75:
        return 3
    else:
        return 4

print(" Generando niveles turístico/local por cluster (GMM)…")

records_niveles = []

for (isla, clus), sub in (
    df[df["cluster_gmm"] != -1]
    .groupby(["isla_fix", "cluster_gmm"])
):
    counts = sub["Tipo_behavior"].value_counts()

    n_fotos_turistico = int(counts.get("Turista", 0))
    n_fotos_local     = int(counts.get("Local", 0))

    p = sub["Tipo_behavior"].value_counts(normalize=True)

    p_turista = p.get("Turista", 0.0)
    p_local   = p.get("Local", 0.0)

    records_niveles.append({
        "isla_fix": isla,
        "cluster_gmm": clus,

        "p_turista": p_turista,
        "p_local": p_local,

        "n_fotos_turistico": n_fotos_turistico,
        "n_fotos_local": n_fotos_local,

        "nivel_turistico": nivel_porcentaje(p_turista),
        "nivel_local": nivel_porcentaje(p_local)
    })

tabla_niveles = pd.DataFrame(records_niveles)

df = df.merge(
    tabla_niveles,
    on=["isla_fix", "cluster_gmm"],
    how="left"
)

gdf_all = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_WGS84
)

gdf_points = gdf_all[gdf_all["cluster_gmm"] != -1].copy()
gdf_noise  = gdf_all[gdf_all["cluster_gmm"] == -1].copy()

def r90_m(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(
        np.clip(
            np.sin(lat0)*np.sin(pts[:,0]) +
            np.cos(lat0)*np.cos(pts[:,0])*np.cos(pts[:,1] - lon0),
            -1.0, 1.0
        )
    )
    return np.percentile(d * 6371000, 90)


centroid_records = []

for (isla, clus), sub in gdf_points.groupby(["isla_fix","cluster_gmm"]):
    centroid_records.append({
        "isla_fix": isla,
        "cluster_gmm": clus,

        "nivel_turistico": sub["nivel_turistico"].iloc[0],
        "nivel_local": sub["nivel_local"].iloc[0],

        "n_fotos_turistico": sub["n_fotos_turistico"].iloc[0],
        "n_fotos_local": sub["n_fotos_local"].iloc[0],

        "lat": sub["latitude"].median(),
        "lon": sub["longitude"].median(),
        "n_fotos": len(sub),
        "r90_m": r90_m(sub)
    })

gdf_centroids = gpd.GeoDataFrame(
    centroid_records,
    geometry=gpd.points_from_xy(
        [c["lon"] for c in centroid_records],
        [c["lat"] for c in centroid_records]
    ),
    crs=CRS_WGS84
)

cent_utm = gdf_centroids.to_crs(CRS_UTM28)
pts_utm  = gdf_points.to_crs(CRS_UTM28)


buffers = []

for _, r in cent_utm.iterrows():
    buffers.append({
        "isla_fix": r["isla_fix"],
        "cluster_gmm": r["cluster_gmm"],

        "nivel_turistico": r["nivel_turistico"],
        "nivel_local": r["nivel_local"],

        "n_fotos_turistico": r["n_fotos_turistico"],
        "n_fotos_local": r["n_fotos_local"],

        "n_fotos": r["n_fotos"],
        "r90_m": r["r90_m"],
        "geometry": r.geometry.buffer(float(r["r90_m"]))
    })

gdf_buffers = gpd.GeoDataFrame(buffers, crs=CRS_UTM28).to_crs(CRS_WGS84)

hulls = []

for (isla, clus), sub in pts_utm.groupby(["isla_fix","cluster_gmm"]):
    geom_union = unary_union(sub.geometry)
    hull = geom_union.convex_hull

    rcent = cent_utm[
        (cent_utm["isla_fix"] == isla) &
        (cent_utm["cluster_gmm"] == clus)
    ].iloc[0]

    hulls.append({
        "isla_fix": isla,
        "cluster_gmm": clus,

        "nivel_turistico": rcent["nivel_turistico"],
        "nivel_local": rcent["nivel_local"],

        "n_fotos_turistico": rcent["n_fotos_turistico"],
        "n_fotos_local": rcent["n_fotos_local"],

        "n_fotos": rcent["n_fotos"],
        "r90_m": rcent["r90_m"],
        "geometry": hull
    })

gdf_hulls = gpd.GeoDataFrame(hulls, crs=CRS_UTM28).to_crs(CRS_WGS84)

spiders = []
cent_idx = {
    (r["isla_fix"], r["cluster_gmm"]): r.geometry
    for _, r in cent_utm.iterrows()
}

for (isla, clus), sub in pts_utm.groupby(["isla_fix","cluster_gmm"]):
    cgeom = cent_idx[(isla, clus)]
    for _, pr in sub.iterrows():
        spiders.append({
            "isla_fix": isla,
            "cluster_gmm": clus,
            "nivel_turistico": pr["nivel_turistico"],
            "nivel_local": pr["nivel_local"],
            "photo_id": pr.get("id", None),
            "geometry": LineString([cgeom, pr.geometry])
        })

gdf_spider = gpd.GeoDataFrame(spiders, crs=CRS_UTM28).to_crs(CRS_WGS84)

Path(OUT_GPKG).parent.mkdir(parents=True, exist_ok=True)

gdf_centroids.to_file(OUT_GPKG, layer="gmm_centroids",      driver="GPKG")
gdf_buffers.to_file(  OUT_GPKG, layer="gmm_buffers_r90",   driver="GPKG")
gdf_hulls.to_file(    OUT_GPKG, layer="gmm_hulls",         driver="GPKG")
gdf_points.to_file(   OUT_GPKG, layer="gmm_points",        driver="GPKG")
gdf_noise.to_file(    OUT_GPKG, layer="gmm_noise",         driver="GPKG")
gdf_spider.to_file(   OUT_GPKG, layer="gmm_spider",        driver="GPKG")

print("✅ Exportado GeoPackage GMM con niveles turístico/local:", OUT_GPKG)

 Generando niveles turístico/local por cluster (GMM)…
✅ Exportado GeoPackage GMM con niveles turístico/local: /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/gmm/gmm_canarias.gpkg
